# Day 07: CUDA Kernels, Kernel Selection & Kernel Fusion
> *100 Days of Inference* | Layer: **Runtime** | Book: *Inference Engineering* Ch 4.1 (pp. 96–101)

**Prerequisite:** Day 06 (ops:byte ratio)

**Goal:** Understand what a CUDA kernel is, how kernel selection works, and why kernel fusion is critical for memory-bound inference.

## What problem does this solve?

Day 04 showed that decode is memory-bound: the bottleneck is reading bytes from GPU memory, not computing. So why does compute efficiency still matter?

Because the *way* you move bytes matters as much as *how many* you move. When two operations run back-to-back — say, a matrix multiply followed by an activation function — the naive approach writes the intermediate result to GPU memory and reads it back for the next operation. That's wasted memory traffic.

CUDA kernels are the functions that run on the GPU. Understanding them is like understanding how system calls work: you don't write them every day, but knowing what's happening under the hood explains why things are fast or slow. Kernel selection and fusion are the two biggest levers for reducing unnecessary memory traffic.

## Concept Overview

**CUDA** (Compute Unified Device Architecture) is NVIDIA's platform for running code on GPUs. A **CUDA kernel** is a function that runs in parallel across thousands of GPU threads simultaneously.

**Infrastructure analogy:** Think of a CUDA kernel like a shell script that runs on every core simultaneously. Your script processes one item; the GPU runs it on 4096 items at once.

Key CUDA components:
- **Kernel:** The parallel function itself (like a thread handler)
- **CUDA driver:** The low-level interface to the GPU hardware (like a device driver)
- **CUDA runtime:** Developer-facing API for launching kernels (like a system call API)
- **cuBLAS:** Pre-built kernels for linear algebra (GEMM = General Matrix-Matrix Multiply)
- **CUTLASS:** Template library for writing high-performance kernels

**Kernel fusion:** Combining two kernels into one to eliminate the intermediate memory read/write — reduces total bytes moved.

In [1]:
!pip install -q numpy matplotlib torch 2>/dev/null
import numpy as np
import matplotlib.pyplot as plt
import torch
import time
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU Memory: 15.6 GB


## Part 1: What a CUDA Kernel Is

A kernel is just a function. Here is the simplest possible one from the book:

```c
__global__ void doubleArray(float *arr, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        arr[i] *= 2.0f;
    }
}
```

This multiplies every element of an array by 2. On a CPU, this runs element-by-element (O(n) time). On a GPU, all elements run simultaneously (O(1) effective time for large n, limited by memory bandwidth).

The `__global__` keyword means it runs on the GPU. The line `int i = blockIdx.x * blockDim.x + threadIdx.x` is the **global thread index formula** — how each thread figures out which array element it owns.

Picture the GPU launching thousands of threads organized into blocks:

```
Block 0          Block 1          Block 2
[t0 t1 t2 t3]   [t4 t5 t6 t7]   [t8 t9 t10 t11]
```

- `blockDim.x` = threads per block (4 here)
- `blockIdx.x` = which block this thread is in (0, 1, 2)
- `threadIdx.x` = position within the block (0–3)

So for thread `t7`: `blockIdx.x=1`, `blockDim.x=4`, `threadIdx.x=3` → `1*4 + 3 = 7`. That thread processes `arr[7]`.

**Infrastructure analogy:** It's like running a bash one-liner via `parallel` across 4096 workers simultaneously. Each worker knows its index (like `$PARALLEL_SEQ`) and processes its slice.

In PyTorch, every operation (add, multiply, matmul, softmax) dispatches one or more CUDA kernels. You don't write them — but they're there.

In [2]:
# Demonstrate kernel dispatch: every PyTorch op is a kernel
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Create a simple tensor on GPU
x = torch.ones(1024, 1024, dtype=torch.float32, device=device)

# Each of these dispatches a CUDA kernel
ops = {
    "Elementwise multiply (x*2)": lambda t: t * 2,
    "ReLU activation":            lambda t: torch.relu(t - 0.5),
    "LayerNorm":                   lambda t: torch.nn.functional.layer_norm(t, t.shape[-1:]),
    "SiLU activation":             lambda t: torch.nn.functional.silu(t),
    "Softmax":                     lambda t: torch.softmax(t, dim=-1),
}

def time_op(fn, tensor, n=100):
    # Warmup
    for _ in range(5):
        fn(tensor)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(n):
        fn(tensor)
        if device.type == 'cuda':
            torch.cuda.synchronize()
    return (time.perf_counter() - start) / n * 1000  # ms

print(f"\nKernel dispatch times (1024x1024 tensor on {device}):")
print(f"{'Operation':<35} {'Latency (ms)':>12}")
print("-" * 50)
for name, fn in ops.items():
    ms = time_op(fn, x)
    print(f"{name:<35} {ms:>10.3f} ms")

Device: cuda

Kernel dispatch times (1024x1024 tensor on cuda):
Operation                           Latency (ms)
--------------------------------------------------
Elementwise multiply (x*2)               0.061 ms
ReLU activation                          0.107 ms
LayerNorm                                0.063 ms
SiLU activation                          0.060 ms
Softmax                                  0.054 ms


## Part 2: The Cost of Separate Kernels (Kernel Launch Overhead + Memory Round-trips)

When two kernels run back-to-back on the same data, there are two costs:

1. **Kernel launch overhead:** Each kernel launch has a small but non-zero overhead (~10-50 microseconds). If you're running hundreds of operations per token, this adds up.

2. **Memory round-trips:** The naive pipeline writes intermediate results to GPU VRAM and reads them back. From Day 04, we know memory access is often the bottleneck.

**Example from the book:** Two kernels: `multiply_by_2` and `multiply_by_3` running sequentially:
1. Read `[1, 2, 3]` from memory
2. Run `×2` → output `[2, 4, 6]`
3. **Write `[2, 4, 6]` to memory** ← wasted I/O
4. **Read `[2, 4, 6]` from memory** ← wasted I/O
5. Run `×3` → output `[6, 12, 18]`
6. Write `[6, 12, 18]` to memory

With a **fused kernel** (`multiply_by_6`):
1. Read `[1, 2, 3]` from memory
2. Run `×6` → output `[6, 12, 18]`
3. Write `[6, 12, 18]` to memory

Steps 3 and 4 are eliminated — 2 fewer memory operations.

In [3]:
# Demonstrate memory round-trip cost with a real example
# We'll simulate a typical LLM FFN activation: linear -> SiLU -> linear

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Scaled down from Llama-7B (4096/11008) to fit DGX Spark memory
batch_seq = 16   # batch * seq_len tokens
hidden = 2048
ffn_dim = 5504

x = torch.randn(batch_seq, hidden, dtype=torch.float16, device=device)
W1 = torch.randn(hidden, ffn_dim, dtype=torch.float16, device=device)
W2 = torch.randn(ffn_dim, hidden, dtype=torch.float16, device=device)
W_gate = torch.randn(hidden, ffn_dim, dtype=torch.float16, device=device)

# Unfused: separate operations, each kernel writes/reads intermediate
def ffn_unfused(x, W1, W_gate, W2):
    hidden_states = torch.matmul(x, W1)      # kernel 1: matmul
    gate = torch.matmul(x, W_gate)           # kernel 2: matmul
    gate_activated = torch.nn.functional.silu(gate)  # kernel 3: silu
    hidden_gated = hidden_states * gate_activated     # kernel 4: elementwise mul
    out = torch.matmul(hidden_gated, W2)     # kernel 5: matmul
    return out

def benchmark(fn, n=50):
    for _ in range(5):
        fn()
    if device.type == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(n):
        fn()
        if device.type == 'cuda':
            torch.cuda.synchronize()
    return (time.perf_counter() - start) / n * 1000

t_unfused = benchmark(lambda: ffn_unfused(x, W1, W_gate, W2))
print(f"FFN unfused (5 kernel dispatches): {t_unfused:.3f} ms")

# torch.compile can automatically fuse kernels
if device.type == 'cuda':
    try:
        ffn_compiled = torch.compile(ffn_unfused, mode='reduce-overhead')
        # Warmup compilation
        _ = ffn_compiled(x, W1, W_gate, W2)
        torch.cuda.synchronize()
        t_compiled = benchmark(lambda: ffn_compiled(x, W1, W_gate, W2))
        print(f"FFN compiled (torch.compile):    {t_compiled:.3f} ms")
        speedup = t_unfused / t_compiled
        print(f"Speedup from compilation:        {speedup:.2f}x")
    except Exception as e:
        print(f"torch.compile not available: {e}")
else:
    print("GPU not available, skipping compile comparison")

# Free memory for subsequent cells
del x, W1, W2, W_gate
if device.type == 'cuda':
    torch.cuda.empty_cache()


FFN unfused (5 kernel dispatches): 0.506 ms


W0413 05:55:46.567000 4333 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


FFN compiled (torch.compile):    1.137 ms
Speedup from compilation:        0.44x


## Part 3: Kernel Selection — Choosing the Right Implementation

**Kernel selection** is choosing the best kernel implementation for a specific operation on specific hardware.

Why does this matter? Kernels are hardware-specific. A kernel optimized for H100's Hopper architecture uses different Tensor Core layouts than one for A100. A kernel that hard-codes VRAM bandwidth values for one GPU will underperform on another.

**Common specialized kernels:**

| Kernel library | What it provides |
|---|---|
| **cuBLAS** | Standard GEMM (matrix multiply) for all NVIDIA GPUs |
| **CUTLASS** | Template library for writing custom GEMM variants |
| **FlashAttention** | Tiled attention that avoids writing S and P matrices to memory |
| **DeepGEMM** | Optimized FP8 GEMM kernels for Hopper GPUs (released by DeepSeek) |
| **FlashInfer** | Optimized attention + sampling kernels for LLM inference |

**Infrastructure analogy:** This is like choosing between `nginx`, `caddy`, and `apache` for your web server. They all serve HTTP, but each has specific performance characteristics for different workloads. You pick based on your use case.

In [4]:
# Demonstrate kernel selection: PyTorch automatically selects the best
# attention kernel based on hardware. We can compare standard vs flash attention.

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def standard_attention(Q, K, V, scale):
    """Standard attention: writes S and P matrices to memory."""
    S = torch.matmul(Q, K.transpose(-2, -1)) * scale  # NxN intermediate
    P = torch.softmax(S, dim=-1)                        # NxN intermediate
    O = torch.matmul(P, V)
    return O

def flash_attention_pytorch(Q, K, V, scale):
    """Uses PyTorch's SDPA which selects FlashAttention when available."""
    return torch.nn.functional.scaled_dot_product_attention(Q, K, V, scale=scale)

# Sized to fit DGX Spark memory — scale up on larger GPUs
batch, heads, d = 2, 4, 64
seq_lengths = [128, 256, 512, 1024]

print(f"Attention benchmark (batch={batch}, heads={heads}, d={d})")
print(f"{'Seq len':>8} {'Standard (ms)':>14} {'Flash/SDPA (ms)':>16} {'Speedup':>8} {'Memory saved':>12}")
print("-" * 65)

for seq_len in seq_lengths:
    Q = torch.randn(batch, heads, seq_len, d, dtype=torch.float16, device=device)
    K = torch.randn(batch, heads, seq_len, d, dtype=torch.float16, device=device)
    V = torch.randn(batch, heads, seq_len, d, dtype=torch.float16, device=device)
    scale = d ** -0.5

    def bench(fn, n=20):
        for _ in range(3): fn(Q, K, V, scale)
        if device.type == 'cuda': torch.cuda.synchronize()
        t = time.perf_counter()
        for _ in range(n):
            fn(Q, K, V, scale)
            if device.type == 'cuda': torch.cuda.synchronize()
        return (time.perf_counter() - t) / n * 1000

    t_std = bench(standard_attention)
    t_flash = bench(flash_attention_pytorch)
    speedup = t_std / t_flash

    # Memory: standard stores NxN matrix (2 of them: S and P)
    n2_mem_mb = 2 * batch * heads * seq_len * seq_len * 2 / 1e6  # FP16
    print(f"{seq_len:>8} {t_std:>12.3f}ms {t_flash:>14.3f}ms {speedup:>7.2f}x {n2_mem_mb:>10.1f} MB")

    del Q, K, V  # free between iterations

if device.type == 'cuda':
    torch.cuda.empty_cache()


Attention benchmark (batch=2, heads=4, d=64)
 Seq len  Standard (ms)  Flash/SDPA (ms)  Speedup Memory saved
-----------------------------------------------------------------
     128        0.147ms          0.060ms    2.47x        0.5 MB
     256        0.148ms          0.080ms    1.84x        2.1 MB
     512        0.236ms          0.158ms    1.49x        8.4 MB
    1024        0.708ms          0.485ms    1.46x       33.6 MB


## Part 4: Profiling Kernel Dispatch with torch.profiler

You can't optimize what you can't measure. PyTorch's built-in profiler shows you exactly which kernels are running and how long each takes.

In [5]:
from torch.profiler import profile, record_function, ProfilerActivity

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
activities = [ProfilerActivity.CPU]
if device.type == 'cuda':
    activities.append(ProfilerActivity.CUDA)

batch, seq, hidden = 4, 128, 2048
x = torch.randn(batch, seq, hidden, dtype=torch.float16, device=device)
W = torch.randn(hidden, hidden * 4, dtype=torch.float16, device=device)

with profile(activities=activities, record_shapes=True) as prof:
    with record_function("FFN_forward"):
        # Simulate a small FFN block
        h = torch.matmul(x, W)                              # linear
        h = torch.nn.functional.silu(h)                     # activation
        h = torch.matmul(h, W.t()[:h.shape[-1]])            # project back
        h = torch.nn.functional.layer_norm(h, [h.shape[-1]]) # norm

# Print top operations by CUDA time
print("Top operations (sorted by self time):")
print("-" * 80)
table = prof.key_averages().table(
    sort_by="self_cuda_time_total" if device.type == 'cuda' else "self_cpu_time_total",
    row_limit=10
)
print(table)

Top operations (sorted by self time):
--------------------------------------------------------------------------------
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                            FFN_forward         0.00%       0.000us         0.00%       0.000us       0.000us       3.799ms       178.66%       3.799ms       3.799ms             1  
                                               aten::mm        27.43%       2.682ms     

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


## Try These Experiments

1. **Measure kernel launch overhead:** Create a loop that launches 1000 tiny elementwise operations (e.g., `x + 1`) on a small tensor. Compare the total time to doing a single equivalent operation. The difference is accumulated launch overhead.

2. **Flash attention memory savings:** In the `standard_attention` function, print the size of the intermediate `S` matrix at different sequence lengths. Calculate how many GB it would consume for a full 32K context window (32768 tokens) with 32 heads.

3. **torch.compile impact:** Try adding `@torch.compile` to `ffn_unfused` for a larger matrix (hidden=8192, ffn=28672, batch_seq=256). Compare latency before and after compilation. What percentage speedup do you get?

## Key Takeaways

- A CUDA **kernel** is a parallel function running on thousands of GPU threads simultaneously — every PyTorch operation dispatches one or more kernels.
- **Kernel fusion** eliminates intermediate memory reads/writes between sequential operations — critical for memory-bound workloads like decode.
- **Kernel selection** picks the best implementation for the hardware. FlashAttention, cuBLAS, and DeepGEMM are specialized for different tasks and GPUs.
- `torch.compile` performs automatic kernel fusion for some patterns; complex kernels like FlashAttention require handwritten fused implementations.
- **What's next:** Day 08 — PyTorch, model file formats, ONNX & TensorRT: how models are stored, compiled, and optimized for deployment.

## References
- *Inference Engineering* Ch 4.1 (pp. 96–101) — CUDA kernels, kernel selection, kernel fusion